# 01 — Data Profiling: Raw Population Excel Files

## Purpose

Profile the raw population Excel workbooks prior to transformation.

## This notebook validates:

* Deterministic programmatic loadability of files

* Stability and completeness of raw schema

* Internal consistency of identifiers, values, and relationships across years

## Findings Summary

### Structural Findings

* Each workbook contains exactly one sheet (always loaded with `sheet_name=0`)

* Sheet names encode a 4-digit year

* Header row is consistently located at row index 1 (`header=1`)

* Column schema evolves across three eras:
  * **2011–2012**: no county code column
  * **2013–2022**: `Megye\nkód`
  * **2023+**: `Vármegye\nkód`

* County name column evolves across two eras:
  * **2011–2022**: `Megye`
  * **2023+**: `Vármegye`

* `Unnamed` columns appear only in **2022** and **2026**, and are fully empty

### Content Findings

* County codes are restricted to the domain `0–20`

* County names are not-nulls, and belong to an identical set of three-vharacter abbreviations

* `KSH\nkód` (settlement identifier):
  * non-null
  * integer type
  * non-negative
  * unique per year

* Settlement names:
  * non-null
  * unique per year
  * whitespace-clean (no unexpected internal spacing except known Budapest cases)

* Settlement type encoding evolves across three eras:
  * **2011–2022**: string labels (Megye-era terminology)
  * **2023–2025**: string labels (Vármegye-era terminology)
  * **2026+**: numeric codes

* Population columns:
  * non-null
  * integer type
  * strictly positive

* Known settlement lifecycle changes are fully captured:
  * **2015**: Balatonakarattya (`34421`) introduced

### Relationship Findings

* Male + female population always equals total population

* `KSH code → settlement name` mapping is stable across years

* `county code → county name` mapping is stable across years

* Settlement-to-county assignment is stable except for known administrative changes:
  * **2014**: Balatonvilágos (`3559`) reassigned VES → SOM
  * **2015**: Balatonakarattya (`34421`) introduced in VES

---

## Setup

In [ ]:
import re

import pandas as pd

from hpm.settings import settings

pd.set_option("display.max_colwidth", None)

# Load raw data
all_xl_files = []

all_xl_files = list(settings.raw_population.iterdir())

## Structural Finding 1 — Single Sheet Per Workbook

In [ ]:
df = pd.DataFrame(
    [
        {"file": xl.name, "sheet_count": len(pd.ExcelFile(xl).sheet_names)}
        for xl in all_xl_files
    ]
)

display(df)

assert (df["sheet_count"] == 1).all(), "❌ Some workbooks have multiple sheets"
print("✅ All workbooks have exactly one sheet")

## Content Finding 1 — Year Encoded in Sheet Name

In [ ]:
df = pd.DataFrame(
    [
        {"file": xl.name, "sheet_name": pd.ExcelFile(xl).sheet_names[0]}
        for xl in all_xl_files
    ]
)

display(df)

assert df["sheet_name"].str.contains(r"\d{4}", regex=True).all(), (
    "❌ Some filenames don't contain a 4-digit year"
)
print("✅ All sheet names contain a 4-digit year")

## Year Mapping

In [ ]:
year_to_xl_files = {}

for xl in all_xl_files:
    sheet = pd.ExcelFile(xl).sheet_names[0]
    year = int(re.search(r"\d{4}", sheet).group())
    year_to_xl_files[year] = xl

year_to_xl_files = dict(sorted(year_to_xl_files.items()))

## Structural Finding 2 — Header Row Position

In [ ]:
for year, xl in year_to_xl_files.items():
    df_raw = pd.read_excel(xl, sheet_name=0, header=None, nrows=2)

    first_row = [str(val) for val in df_raw.iloc[0].dropna()]
    second_row = [str(val) for val in df_raw.iloc[1].dropna()]

    assert len(second_row) > len(first_row), (
        f"❌ {year}: header row is not on second row"
    )

print("✅ Header row consistently located on second row")

## Discover Headers

In [ ]:
df_all_list = []

for year, xl in year_to_xl_files.items():
    df_raw = pd.read_excel(xl, sheet_name=0, header=1, index_col=None, nrows=1)
    df_all_list.append({"year": year, "header": list(df_raw.columns)})

df = pd.DataFrame(df_all_list)

display(df)

## Structural Finding 3 — Empty Column Validation

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    unnamed_cols = [col for col in df.columns if "Unnamed" in str(col)]
    if unnamed_cols:
        assert df[unnamed_cols].isna().all().all(), (
            f"❌ {year}: unnamed columns contain data"
        )

print("✅ All unnamed columns are fully empty")

## Structural Finding 4 — County Code Schema Evolution
 * 2011-2012: absent
 * 2013-2022: present as `Megye\nkód`
 * 2023-2026: present as `Vármegye\nkód`

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    if year <= 2012:
        assert "Megye\nkód" not in df.columns and "Vármegye\nkód" not in df.columns, (
            f"❌ {year}: county code column should be absent"
        )
    elif year <= 2022:
        assert "Megye\nkód" in df.columns, f"❌ {year}: expected Megye\nkód"
    else:
        assert "Vármegye\nkód" in df.columns, f"❌ {year}: expected Vármegye\nkód"

print("✅ County code column follows expected temporal evolution")

## Structural Finding 5 — County Name Schema Evolution
 * 2011-2022: `Megye`
 * 2023+: `Vármegye`

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    if year <= 2022:
        assert "Megye" in df.columns, f"❌ {year}: expected Megye"
    else:
        assert "Vármegye" in df.columns, f"❌ {year}: expected Vármegye"

print("✅ County name column follows expected evolution")

## Content Finding 2 — County Code Validity

In [ ]:
KNOWN_COUNTY_CODES = set(range(0, 21))

for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    county_col = (
        "Megye\nkód"
        if "Megye\nkód" in df.columns
        else "Vármegye\nkód"
        if "Vármegye\nkód" in df.columns
        else None
    )

    if county_col:
        actual = set(df[county_col].unique().astype(int).tolist())
        print(f"{year}: {actual}")
        assert actual == KNOWN_COUNTY_CODES, (
            f"❌ {year}: unexpected county codes: {actual - KNOWN_COUNTY_CODES}"
        )

print("✅ County codes are exactly 0-20 across all years")

## Content Finding 3 — County Name Completeness

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    col = "Megye" if year <= 2022 else "Vármegye"

    assert df[col].isna().sum() == 0, f"❌ {year}: county names contain nulls"

print("✅ County names are always non-null")

## Content Finding 4 — County Name Integrity

In [ ]:
county_name_sets = {}

for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    col = "Megye" if year <= 2022 else "Vármegye"

    county_name_sets[year] = set(df[col].unique())

# Compare all years to first year
base_year = min(county_name_sets)
base_set = county_name_sets[base_year]

for year, s in county_name_sets.items():
    print(f"{year}: {sorted(s)}")
    assert s == base_set, f"❌ {year}: County name mismatch: {s ^ base_set}"

print("✅ County names form an identical set across all years")

## Content Finding 5 — HCSO Code Integrity

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    ksh = df["KSH\nkód"]

    assert ksh.isna().sum() == 0, f"❌ {year}: HCSO code has nulls"
    assert ksh.dtype == "int64", f"❌ {year}: HCSO code is not int64"
    assert (ksh >= 0).all(), f"❌ {year}: HCSO code has negative values"

print("✅ HCSO codes are nonnegative integers")

## Content Finding 6 — HCSO Code Uniqueness

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    assert df["KSH\nkód"].duplicated().sum() == 0, (
        f"❌ {year}: duplicate HCSO codes found"
    )

print("✅ HCSO codes are unique per year")

## Content Finding 7 — Settlement Name Integrity

In [ ]:
changes = {}

for (year_a, xl_a), (year_b, xl_b) in zip(
    list(year_to_xl_files.items()), list(year_to_xl_files.items())[1:]
):
    df_a = pd.read_excel(xl_a, sheet_name=0, header=1)
    df_b = pd.read_excel(xl_b, sheet_name=0, header=1)

    codes_a = set(df_a["KSH\nkód"].tolist())
    codes_b = set(df_b["KSH\nkód"].tolist())

    added = codes_b - codes_a
    removed = codes_a - codes_b

    if added or removed:
        changes[year_b] = {"added": added, "removed": removed}

print(changes)

In [ ]:
KNOWN_CHANGES = {
    2015: {"added": {34421}, "removed": set()},  # Balatonakarattya
}

for (year_a, xl_a), (year_b, xl_b) in zip(
    list(year_to_xl_files.items()), list(year_to_xl_files.items())[1:]
):
    df_a = pd.read_excel(xl_a, sheet_name=0, header=1)
    df_b = pd.read_excel(xl_b, sheet_name=0, header=1)

    codes_a = set(df_a["KSH\nkód"].tolist())
    codes_b = set(df_b["KSH\nkód"].tolist())

    added = codes_b - codes_a
    removed = codes_a - codes_b

    expected_added = KNOWN_CHANGES.get(year_b, {}).get("added", set())
    expected_removed = KNOWN_CHANGES.get(year_b, {}).get("removed", set())

    assert added == expected_added, (
        f"❌ {year_a}→{year_b}: unexpected new settlements: {added - expected_added}"
    )
    assert removed == expected_removed, (
        f"❌ {year_a}→{year_b}: unexpected removed settlements: {removed - expected_removed}"
    )

print("✅ All settlement changes between years are known and accounted for")

## Content Finding 8 — Settlement Name Without Nulls

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    assert df["Település"].isna().sum() == 0, f"❌ {year}: Settlement name has nulls"

print("✅ Settlement name is always non-null across all years")

## Content Finding 9 — Settlement Name Uniqueness

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    assert df["Település"].duplicated().sum() == 0, (
        f"❌ {year}: duplicate settlement names found"
    )

print("✅ Settlement names are unique per year across all years")

## Content Finding 10 — Settlement Name Redundant Whitespace

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    whitespace_names = df[df["Település"].str.contains(r"\s{2,}", regex=True)][
        "Település"
    ]
    non_budapest = [
        name for name in whitespace_names if not name.startswith("Budapest")
    ]
    assert len(non_budapest) == 0, (
        f"❌ {year}: unexpected whitespace in non-Budapest names: {non_budapest}"
    )

print(
    "✅ Excessive whitespace is exclusive to Budapest district names across all years"
)

## Discover Settlement Type Content

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    print(f"{year}: {df['Település\ntípusa'].unique().tolist()}")

## Content Finding 11 — Settlement Type Evolution
Values evolved across three eras requiring normalization:
- 2011-2022: string labels (Megye era)
- 2023-2025: string labels with Megye → Vármegye rename
- 2026+: numeric codes (documented in cell notes of Excel file)
  - 0 = LAKCÍM NÉLKÜLI
  - 1 = fővárosi kerület
  - 2 = vármegye székhely
  - 3 = vármegyei jogú város
  - 6 = város
  - 7 = nagyközség
  - 8 = község

### Anomalies
- 2013: typo — 'megye jogú város' instead of 'megyei jogú város'
- 2017: 0 encoded as integer instead of string

In [ ]:
KNOWN_SETTLEMENT_TYPES = {
    "megye": {
        "0",
        "fővárosi kerület",
        "megye székhely",
        "megyei jogú város",
        "megye jogú város",  # 2013 typo
        "város",
        "nagyközség",
        "község",
    },
    "vármegye": {
        "0",
        "fővárosi kerület",
        "vármegye székhely",
        "vármegyei jogú város",
        "város",
        "nagyközség",
        "község",
    },
    "numeric": {0, 1, 2, 3, 6, 7, 8},
}

for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)

    # normalize 0 int to "0" string before checking - 2017 integer 0 inconsistency
    actual = set(df["Település\ntípusa"].astype(str).unique().tolist())

    if int(year) <= 2022:
        expected = {str(v) for v in KNOWN_SETTLEMENT_TYPES["megye"]}
    elif int(year) <= 2025:
        expected = {str(v) for v in KNOWN_SETTLEMENT_TYPES["vármegye"]}
    else:
        expected = {str(v) for v in KNOWN_SETTLEMENT_TYPES["numeric"]}

    unexpected = actual - expected
    assert not unexpected, f"❌ {year}: unexpected settlement types: {unexpected}"

print("✅ Settlement types follow expected era evolution across all years")

## Content Finding 12 — Population Validity

In [ ]:
POP_COLS = [
    "Állandó férfi\nlakosság összesen",
    "Állandó női\nlakosság összesen",
    "Állandó lakosság\nösszesen",
]

for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    for col in POP_COLS:
        assert df[col].isna().sum() == 0, f"❌ {year} | {col}: has nulls"
        assert df[col].dtype == "int64", f"❌ {year} | {col}: not int64"
        assert (df[col] > 0).all(), f"❌ {year} | {col}: has non-positive values"

print("✅ All population columns are non-null, positive integers")

## Relationship Finding 1 — Population Consistency

In [ ]:
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    assert (
        df["Állandó férfi\nlakosság összesen"] + df["Állandó női\nlakosság összesen"]
        == df["Állandó lakosság\nösszesen"]
    ).all(), f"❌ {year}: male + female != total population"

print("✅ Male + female equals total population")

## Relationship Finding 2 — HCSO Code → Settlement Name Stability

In [ ]:
name_maps = {}
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    name_maps[year] = df.set_index("KSH\nkód")["Település"].to_dict()

years = list(name_maps.keys())
for year_a, year_b in zip(years, years[1:]):
    for ksh, name_b in name_maps[year_b].items():
        name_a = name_maps[year_a].get(ksh)
        if name_a is not None and name_b != name_a:
            assert False, f"❌ {year_a}→{year_b}: KSH {ksh} changed '{name_a}' → '{name_b}'"

print("✅ HCSO code → settlement name mapping is stable")

## Relationship Finding 3 — County Code → County Name Stability

In [ ]:
county_code_name_maps = {}
for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    county_col = "Megye" if int(year) <= 2022 else "Vármegye"
    code_col = "Megye\nkód" if "Megye\nkód" in df.columns else "Vármegye\nkód" if "Vármegye\nkód" in df.columns else None
    if code_col:
        county_code_name_maps[year] = df.set_index(code_col)[county_col].to_dict()

years = list(county_code_name_maps.keys())
for year_a, year_b in zip(years, years[1:]):
    for code, name_b in county_code_name_maps[year_b].items():
        name_a = county_code_name_maps[year_a].get(code)
        assert name_a is None or name_b == name_a, f"❌ {year_a}→{year_b}: county code {code} changed '{name_a}' → '{name_b}'"

print("✅ County code → county name mapping is stable")

## Relationship Finding 4 — Settlement → County Stability
- KSH 34421 (Balatonakarattya): absent until 2015, then appears as VES
- KSH 3559 (Balatonvilágos): moved from VES to SOM in 2014 (administrative boundary change)

In [ ]:
county_maps = {}

for year, xl in year_to_xl_files.items():
    df = pd.read_excel(xl, sheet_name=0, header=1)
    county_col = "Megye" if int(year) <= 2022 else "Vármegye"
    mapping = df.groupby("KSH\nkód")[county_col].first().to_dict()
    county_maps[year] = mapping

county_reassignments = {}

years = list(county_maps.keys())
for year_a, year_b in zip(years, years[1:]):
    for ksh, county_b in county_maps[year_b].items():
        county_a = county_maps[year_a].get(ksh)
        if county_b != county_a:
            if year_b not in county_reassignments:
                county_reassignments[year_b] = {}
            county_reassignments[year_b][ksh] = (county_a, county_b)

print(county_reassignments)

In [ ]:
KNOWN_COUNTY_REASSIGNMENTS = {
    2014: {3559: ("VES", "SOM")},  # Balatonvilágos moved from Veszprém to Somogy
    2015: {34421: (None, "VES")},  # Balatonakarattya new settlement in Veszprém
}

years = list(county_maps.keys())
for year_a, year_b in zip(years, years[1:]):
    for ksh, county_b in county_maps[year_b].items():
        county_a = county_maps[year_a].get(ksh)
        if county_a and county_b != county_a:
            expected = KNOWN_COUNTY_REASSIGNMENTS.get(year_b, {})
            assert ksh in expected, f"❌ {year_a}→{year_b}: unexpected county change for KSH {ksh}: {county_a} → {county_b}"

print("✅ All county reassignments between years are known and accounted for")